In [2]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

11499
2253216


In [3]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset, collate_fun_emb
import numpy as np


def collate_fun_conv(batch):
    mzs_intens = []
    for mz, inten in batch:
        mz_inten = np.zeros(1000, dtype=np.float32)
        mz_inten[mz] = inten
        mzs_intens.append(mz_inten)
    return pt.tensor(np.array(mzs_intens), dtype=pt.float32)


dataset_lib = SpecDataset(mols_all)
dataset_test = SpecDataset(mols_test)
loader_lib1 = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
loader_test1 = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
loader_lib2 = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
loader_test2 = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)

In [5]:
from utils.model import Spec2Emb
import torch.nn as nn


class SpecConv(nn.Module):
    def __init__(self, spec_len:int=1000):
        super(SpecConv, self).__init__()
        self.linear1 = nn.Linear(spec_len, 500)
        self.linear2 = nn.Linear(500, 250)
        self.linear3 = nn.Linear(250, 125)
        self.encoder = nn.Sequential(self.linear1, nn.ReLU(), self.linear2, nn.ReLU(), self.linear3)
        self.linear4 = nn.Linear(125, 250)
        self.linear5 = nn.Linear(250, 500)
        self.linear6 = nn.Linear(500, spec_len)
        self.decoder = nn.Sequential(self.linear4, nn.ReLU(), self.linear5, nn.ReLU(), self.linear6)

    def forward(self, mzs_intens, mode:str='train'):
        if mode == 'train': 
            return self.decoder(self.encoder(mzs_intens))
        elif mode == 'emb': # emb模式下的masks只mask掉了padding 
            return self.encoder(mzs_intens)
        else:
            raise ValueError('mode not exist')
        
gpu = 9
model1 = SpecConv().to(gpu)
model1.load_state_dict(pt.load('./model/Linear_epoch5.pth', map_location='cpu'))
model2 = Spec2Emb().to(gpu)
model2.load_state_dict(pt.load('./model/base_peak0.01_epoch4.pth', map_location='cpu'))

<All keys matched successfully>

In [6]:
from utils.tools import build_idx, evaluate, gen_embeddings


def gen_embeddings1(model:nn.Module, loader:DataLoader, gpu:int):
    model.eval()
    embs = []
    with pt.no_grad():
        for mzs_intens in loader:
            data = mzs_intens.to(gpu)
            emb = model(data, mode='emb').detach().cpu().numpy()
            embs.append(emb)
    pt.cuda.empty_cache()
    embs = np.concatenate(embs, axis=0)
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


f = open('merge.txt', 'w')
embs_lib1 = gen_embeddings1(model1, loader_lib1, gpu)
embs_test1 = gen_embeddings1(model1, loader_test1, gpu)
embs_lib2 = gen_embeddings(model2, loader_lib2, gpu)
embs_test2 = gen_embeddings(model2, loader_test2, gpu)
embs_lib = np.concatenate([embs_lib1, embs_lib2], axis=1)
embs_test = np.concatenate([embs_test1, embs_test2], axis=1)
I_expand, _ = build_idx(embs_lib, embs_test, gpu)
top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'Expanded')
I_insilico, _ = build_idx(embs_lib[:2146690], embs_test, gpu)
top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'In-silico')

Searching time:  0:00:01.927017
Expanded library
Top1 hit rate: 21.09%
Top10 hit rate: 55.34%
Searching time:  0:00:01.771899
In-silico library
Top1 hit rate: 21.34%
Top10 hit rate: 55.74%
